# ScaleRAG – Multimodal RAG Data Preparation Pipeline
---
This notebook walks through a **complete end-to-end pipeline** that transforms raw research PDFs into **structured, multimodal datasets** ready for **Retrieval-Augmented Generation (RAG)** applications.

It covers every stage of the preprocessing workflow — from downloading PDFs to extracting structured text, figures, tables, and equations, followed by enrichment with cropped visual assets and chunk merging for optimized retrieval.

### Pipeline Overview
1. **Setup & Imports** – Initialize dependencies and utility scripts.
2. **Download PDFs** → `data/pdf/` – Fetch all papers listed in `core_papers.csv`.
3. **Parser Prototype (v1)** – Early rule-based multimodal parser (for reference).
4. **PDF → Docling JSON** – Structured document conversion via Docling.
5. **Docling → RAG JSON** – Convert structured representations into LLM-ready blocks.
6. **Enrich RAG with Images** – Attach figure/table crops to RAG blocks.
7. **Merge Short Chunks** – Consolidate fragmented text for coherent retrieval units.

All stages are **idempotent** — rerunning the notebook is safe; existing files are automatically skipped.

---

### Output Directory Summary

| Stage | Directory | Description |
|:------|:-----------|:-------------|
| Raw PDFs | `data/pdf/` | Downloaded research papers |
| Docling JSONs | `data/docling_json*/` | Structured outputs from Docling |
| RAG JSONs | `data/rag_json*/` | LLM-ready RAG blocks |
| Cropped Images | `data/rag_assets/images/` | Extracted figures & tables |
| Final Chunks | `data/rag_chunks/` | Clean, merged text chunks for retrieval |

---

This notebook forms the **core preprocessing pipeline** for the ScaleRAG project, enabling **text + image multimodal retrieval** for large-language-model research.


In [1]:
# imports
import torch

import numpy as np

import pandas as pd

## Step 1:  Download Research Papers from arXiv

This step automatically downloads all research papers listed in the file **`core_papers.csv`**.  
The CSV contains at least one column named `arxiv_id`, and optionally a `title` column for readability.

The helper function `download_all_papers()` performs the following tasks:

1. **Read manifest file** — Loads all arXiv IDs from `core_papers.csv`.  
2. **Validate IDs** — Ensures each entry follows the correct `YYYY.NNNNN` arXiv format.  
3. **Download missing PDFs** — Fetches each paper from arXiv (`https://arxiv.org/pdf/<id>.pdf`).  
   - Skips files that already exist in the target folder to avoid duplicates.  
4. **Save output** — Stores all PDFs inside the directory `data/pdf/`.


In [3]:
from utils.data import download_all_papers

download_all_papers()

[INFO] File data/pdf/2001.08361.pdf already exists.
[INFO] File data/pdf/2203.15556.pdf already exists.
[INFO] File data/pdf/2005.03141.pdf already exists.
[INFO] File data/pdf/2306.10209.pdf already exists.
[INFO] File data/pdf/2307.08691.pdf already exists.
[INFO] File data/pdf/2312.00752.pdf already exists.
[INFO] File data/pdf/2309.06180.pdf already exists.
[INFO] File data/pdf/2406.03243.pdf already exists.
[INFO] File data/pdf/2211.17192.pdf already exists.
[INFO] File data/pdf/2401.10774.pdf already exists.
[INFO] File data/pdf/2211.10438.pdf already exists.
[INFO] File data/pdf/2306.00978.pdf already exists.
[INFO] File data/pdf/2306.14048.pdf already exists.
[INFO] File data/pdf/2310.01801.pdf already exists.
[INFO] File data/pdf/2101.03961.pdf already exists.
[INFO] File data/pdf/2106.06967.pdf already exists.
[INFO] File data/pdf/2408.03314.pdf already exists.
[INFO] File data/pdf/2408.00724.pdf already exists.
[INFO] File data/pdf/2303.11312.pdf already exists.
[INFO] File 

## Parser Version 1 — Custom Parsing Prototype

This section shows our **first experimental parser setup** (before adopting Docling).  
Here we combined **four separate custom parsers**:
- `text_parser.py` — extracted textual sections and headings using PyMuPDF heuristics  
- `table_parser.py` — located and extracted tables with Camelot + pdfplumber  
- `figure_parser.py` — detected figure captions and cropped figure images  
- `equation_parser.py` — isolated displayed equations and saved them as cropped images  

Each parser was tuned with its own logic for bounding boxes, captions, and heuristics to classify content.  
While this version worked reasonably well, it required heavy rule-based tuning and frequent manual fixes.  

Later, we migrated the entire pipeline to **Docling**.

So this section remains here as a **reference for Version 1**, showing how I first approached multimodal content extraction before switching to the more robust Docling-based method.


In [14]:
# Parser V1 (4 diffreent parsers)

import os, json, fitz

from utils1.text_parser import extract_text_sections
from utils1.figure_parser import extract_figures
from utils1.table_parser import extract_tables
from utils1.equation_parser import extract_equations
from utils1.utils import prepare_output_dir

# === Settings ===
pdf_path = "2001.08361.pdf"
doc_id = os.path.splitext(os.path.basename(pdf_path))[0]
doc = fitz.open(pdf_path)
output_dir = prepare_output_dir("parses", doc_id)

# 1. Parse text ===
result = extract_text_sections(doc, doc_id)

# 2. Parse figures ===
extract_figures(doc, doc_id, output_dir, result)

# 3. Parse tables ===
extract_tables(doc, doc_id, output_dir, pdf_path, result)

# 4. Parse equations ===
extract_equations(doc, doc_id, output_dir, result)

# 5. Save final combined JSON ===
with open(os.path.join(output_dir, f"{doc_id}.json"), "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"All done! Parsed content saved to: {output_dir}/{doc_id}.json")


/home/mt3846/envTorch124/lib/python3.10/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (290.328, 203.8780828, 358.44735799999995, 346.90899260000003)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


All done! Parsed content saved to: parses/2001.08361/2001.08361.json


## Quick Docling Conversion Demo

This cell demonstrates how to convert a research paper directly from an online PDF URL (e.g., arXiv) into structured document formats using **Docling**.

The process involves the following steps:

1. **Initialize the converter** — Create an instance of `DocumentConverter()` from `docling.document_converter`.  
2. **Run the conversion** — Provide a PDF URL to `converter.convert()` to process the document.  
3. **Extract structured data** —  
   - `document.export_to_markdown()` generates a Markdown representation (for readability).  
   - `document.export_to_dict()` produces a JSON-like dictionary suitable for downstream RAG processing.  
4. **Preview the results** — Print the Markdown output to visualize the extracted structure and content.

This simple demo helps validate that the Docling pipeline correctly parses text, sections, tables, and figures before batch-processing the entire dataset.


In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

result = converter.convert("https://arxiv.org/pdf/2408.09869")

document = result.document
markdown_output = document.export_to_markdown()
json_output = document.export_to_dict()

print(markdown_output)



## Step 2: Convert PDFs to Docling JSON

This step runs the helper function **`batch_convert_pdfs()`** from `utils/docling_converter.py`,  
which transforms all downloaded PDFs into structured **Docling JSON** files.

The conversion process uses **Docling’s `DocumentConverter`** engine to parse research papers into a
machine-readable format that preserves text, figures, tables, equations, and layout hierarchy.

### What happens here:
1. **Scan the input folder** (`data/pdf/`) for all `.pdf` files.  
2. **Run the Docling converter** on each file.  
3. **Export a structured dictionary** with every element (sections, paragraphs, captions, etc.).  
4. **Save results** as `.docling.json` files in the output directory (`data/docling_json/`).  
5. **Log progress** — prints success or failure for each paper and a final summary.


Each JSON file now contains the **full structured representation** of a paper and will be used in the next step (Docling → RAG conversion) to create multimodal RAG blocks.


In [61]:
from utils.docling_converter import batch_convert_pdfs

summary = batch_convert_pdfs("data/pdf", "data/docling_json1")

2025-10-28 02:11:58,469 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-28 02:11:58,479 - INFO - Going to convert document batch...
2025-10-28 02:11:58,480 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 4f2edc0f7d9bb60b38ebfecf9a2609f5
2025-10-28 02:11:58,481 - INFO - Accelerator device: 'cuda:0'
[INFO] 2025-10-28 02:11:58,506 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-10-28 02:11:58,519 [RapidOCR] download_file.py:60: File exists and is valid: /home/mt3846/envTorch124/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-10-28 02:11:58,520 [RapidOCR] main.py:53: Using /home/mt3846/envTorch124/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-10-28 02:11:58,601 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-10-28 02:11:58,604 [RapidOCR] download_file.py:60: File exists and is valid: /home/mt3846/envTorch124/lib/python3.10/site-packages/rapido

Found 23 PDFs in /home/mt3846/ScaleRAG-Multimodal-Hierarchical/data/pdf


[INFO] 2025-10-28 02:11:58,671 [RapidOCR] download_file.py:60: File exists and is valid: /home/mt3846/envTorch124/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[INFO] 2025-10-28 02:11:58,672 [RapidOCR] main.py:53: Using /home/mt3846/envTorch124/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
2025-10-28 02:11:58,766 - INFO - Auto OCR model selected rapidocr with onnxruntime.
2025-10-28 02:11:58,766 - INFO - Accelerator device: 'cuda:0'
2025-10-28 02:11:59,684 - INFO - Accelerator device: 'cuda:0'
2025-10-28 02:12:00,640 - INFO - Processing document 2001.08361.pdf
2025-10-28 02:12:20,688 - INFO - Finished converting document 2001.08361.pdf in 22.22 sec.
2025-10-28 02:12:20,778 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-28 02:12:20,781 - INFO - Going to convert document batch...
2025-10-28 02:12:20,782 - INFO - Processing document 2005.03141.pdf


 +++ Saved: 2001.08361.docling.json


2025-10-28 02:12:26,583 - INFO - Finished converting document 2005.03141.pdf in 5.81 sec.
2025-10-28 02:12:26,624 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-28 02:12:26,631 - INFO - Going to convert document batch...
2025-10-28 02:12:26,632 - INFO - Processing document 2101.03961.pdf


...

 +++ Saved: 2408.00724.docling.json


2025-10-28 02:18:43,512 - INFO - Finished converting document 2408.03314.pdf in 35.39 sec.
2025-10-28 02:18:43,593 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


 +++ Saved: 2408.03314.docling.json


2025-10-28 02:18:43,787 - INFO - Going to convert document batch...
2025-10-28 02:18:43,788 - INFO - Processing document 2502.04524.pdf
2025-10-28 02:21:35,206 - INFO - Finished converting document 2502.04524.pdf in 171.62 sec.


 +++ Saved: 2502.04524.docling.json

Done. Success: 23, Failed: 0
JSONs saved in: /home/mt3846/ScaleRAG-Multimodal-Hierarchical/data/docling_json1


## Step 3: Convert Docling JSON → RAG JSON

This stage performs the **final transformation** from structured **Docling JSON** files into lightweight,  
LLM-ready **RAG (Retrieval-Augmented Generation) JSON blocks**.

The helper function `batch_convert_docling_to_rag()` (defined in `utils/docling_to_rag.py`) processes every `.docling.json`
file inside the directory `data/docling_json/` and creates corresponding `.rag.json` outputs in `data/rag_json/`.

### What this step does
1. **Parse each Docling file** — Reads hierarchical structures (`texts`, `tables`, `pictures`, `groups`, etc.).  
2. **Extract multimodal content** — Builds individual **RAG blocks** for:
   - Paragraphs  
   - Equations  
   - Figures (with captions)  
   - Tables (with captions)  
3. **Attach metadata** — Each block stores its `page`, `section`, and bounding box (`bbox`) coordinates.  
4. **Skip irrelevant items** — Page headers, footers, and empty fragments are ignored.  
5. **Output schema** — Every RAG block follows the standard format:

   ```json
   {
     "id": "2106.06967_paragraph_1",
     "type": "paragraph",
     "content": "In this paper we derived in QCD the BFKL linear, inhomogeneous equation ...",
     "metadata": {
       "page": 1,
       "section": "Abstract",
       "bbox": [108.79, 564.82, 507.30, 638.50]
     }
   }

6. Save results — Each converted file is stored as <paper_id>.rag.json (e.g., 2106.06967.rag.json) inside data/rag_json/.

These .rag.json files are now ready to be indexed or embedded for multimodal RAG pipelines, allowing efficient retrieval
of relevant text, figure captions, tables, and equations by large-language-model systems.



In [62]:
from utils.docling_to_rag import batch_convert_docling_to_rag

summary = batch_convert_docling_to_rag("data/docling_json", "data/rag_json1")
print(summary)

Found 23 Docling JSON files in 'data/docling_json'.


Converting Docling → RAG:  30%|███       | 7/23 [00:00<00:00, 62.49file/s]

Converted 2306.00978.docling.json → 2306.00978.rag.json (176 blocks)
Converted 2303.11312.docling.json → 2303.11312.rag.json (237 blocks)
Converted 2211.17192.docling.json → 2211.17192.rag.json (149 blocks)
Converted 2312.11514.docling.json → 2312.11514.rag.json (237 blocks)
Converted 2307.08691.docling.json → 2307.08691.rag.json (124 blocks)
Converted 2101.03961.docling.json → 2101.03961.rag.json (286 blocks)
Converted 2408.03314.docling.json → 2408.03314.rag.json (204 blocks)
Converted 2309.06180.docling.json → 2309.06180.rag.json (209 blocks)
Converted 2310.01801.docling.json → 2310.01801.rag.json (112 blocks)
Converted 2408.00724.docling.json → 2408.00724.rag.json (190 blocks)
Converted 2005.03141.docling.json → 2005.03141.rag.json (92 blocks)
Converted 2306.14048.docling.json → 2306.14048.rag.json (678 blocks)
Converted 2401.18059.docling.json → 2401.18059.rag.json (192 blocks)


Converting Docling → RAG:  87%|████████▋ | 20/23 [00:00<00:00, 55.34file/s]

Converted 2001.08361.docling.json → 2001.08361.rag.json (331 blocks)
Converted 2211.10438.docling.json → 2211.10438.rag.json (144 blocks)
Converted 2406.03243.docling.json → 2406.03243.rag.json (221 blocks)
Converted 2306.10209.docling.json → 2306.10209.rag.json (190 blocks)
Converted 2401.10774.docling.json → 2401.10774.rag.json (204 blocks)
Converted 2303.11313.docling.json → 2303.11313.rag.json (173 blocks)
Converted 2312.00752.docling.json → 2312.00752.rag.json (453 blocks)
Converted 2203.15556.docling.json → 2203.15556.rag.json (226 blocks)


Converting Docling → RAG: 100%|██████████| 23/23 [00:00<00:00, 54.96file/s]

Converted 2502.04524.docling.json → 2502.04524.rag.json (141 blocks)
Converted 2106.06967.docling.json → 2106.06967.rag.json (256 blocks)
{'total_files': 23, 'output_dir': 'data/rag_json1'}


## Step 4: Enrich RAG JSONs with Extracted Images

In this step, we enhance the RAG dataset by **embedding visual information** directly into the existing RAG JSON files.  
The function `enrich_rag_with_images()` from `utils/pdf_image_extractor.py` attaches **cropped figure and table images** to their corresponding RAG entries based on bounding boxes from Docling metadata.

### What this step does

1. **Iterate through RAG JSONs** — Scans all `.rag.json` files in `data/rag_json/`.  
2. **Match captions to visual objects** — For each figure or table block, the script searches Docling metadata to find the best caption match using text-similarity scoring.  
3. **Crop image regions** — When a match is found, it uses PyMuPDF (`fitz`) to crop the figure/table region from the original PDF page.  
4. **Save cropped images** — Each extracted image is saved under:  
data/rag_assets/images/<paper_id>/<block_type>/<figure_or_table>_<n>.png
    
5. **Update RAG metadata** — Adds a new key `image_path` to the corresponding RAG block:
    ```json
    {
      "id": "2309.06180_fig_1",
      "type": "figure",
      "content": "Figure 1. Left: Memory layout when serving an LLM with 13B parameters ...",
      "metadata": {
        "page": 1,
        "section": "Introduction",
        "bbox": [323.74, 483.94, 545.49, 609.22],
        "image_path": "data/rag_assets/images/2309.06180/figure/figure_1.png"
      }
    }

6. Result

After execution, all RAG blocks containing figures or tables now include accurate image paths that link to automatically cropped visuals.
This enriched dataset makes the RAG pipeline multimodal, enabling future retrieval systems to use both text and images for contextual understanding and visualization.


In [63]:
from utils.pdf_image_extractor import enrich_rag_with_images

summary = enrich_rag_with_images()
print(summary)


Enriching RAG JSONs with images:   4%|▍         | 1/23 [00:01<00:35,  1.61s/it]

Enriched 2001.08361.rag.json


Enriching RAG JSONs with images:   9%|▊         | 2/23 [00:01<00:17,  1.19it/s]

Enriched 2005.03141.rag.json


Enriching RAG JSONs with images:  13%|█▎        | 3/23 [00:02<00:16,  1.23it/s]

...

Enriched 2408.00724.rag.json


Enriching RAG JSONs with images:  96%|█████████▌| 22/23 [00:23<00:01,  1.05s/it]

Enriched 2408.03314.rag.json


Enriching RAG JSONs with images: 100%|██████████| 23/23 [00:29<00:00,  1.29s/it]

Enriched 2502.04524.rag.json
Done. Updated 23 RAG JSON files with image paths.
{'updated_files': 23, 'rag_dir': 'data/rag_json1'}


### Step 5: Chunking and Text Merging Overview

This cell runs the **`merge_short_text_chunks()`** function from `utils/chunk_merger.py`, which processes all `.json` files inside `data/rag_json/` and produces merged output files in `data/rag_chunks/`.

The script’s purpose is to **merge very short consecutive text entries** (e.g., tiny paragraphs, headers, or section titles) into longer, coherent text chunks — while keeping **non-text elements** such as *figures, tables, and equations* separate.

#### How It Works
1. **Iterates through all `.json` documents** under `data/rag_json/`.
2. **Reads each entry** and checks its type:
   - If it’s a `paragraph`, `section`, or `header` shorter than ~50 characters or with fewer than 10 words, it’s treated as a “short chunk.”
   - Such short chunks are **merged** with the next neighboring text until they form a complete sentence ending with punctuation (`.`, `?`, `!`).
3. **Non-text items** (like figures/tables) are preserved as-is and not merged.
4. **Outputs a new JSON file** for each paper, named `<paper_id>.rag.chunks.json`.
5. **Reports total merged chunk counts** both per file and overall.


This ensures cleaner and more semantically consistent RAG chunks for downstream retrieval or LLM processing.


In [67]:
from utils.chunk_merger import merge_short_text_chunks

merge_short_text_chunks()

✅ 2310.01801.rag.json: 112 chunks saved to 2310.01801.rag.chunks.json
✅ 2306.14048.rag.json: 661 chunks saved to 2306.14048.rag.chunks.json
✅ 2307.08691.rag.json: 118 chunks saved to 2307.08691.rag.chunks.json
✅ 2408.00724.rag.json: 190 chunks saved to 2408.00724.rag.chunks.json
✅ 2303.11313.rag.json: 173 chunks saved to 2303.11313.rag.chunks.json
✅ 2406.03243.rag.json: 221 chunks saved to 2406.03243.rag.chunks.json
✅ 2312.11514.rag.json: 234 chunks saved to 2312.11514.rag.chunks.json
✅ 2306.00978.rag.json: 173 chunks saved to 2306.00978.rag.chunks.json
✅ 2401.10774.rag.json: 203 chunks saved to 2401.10774.rag.chunks.json
✅ 2211.17192.rag.json: 145 chunks saved to 2211.17192.rag.chunks.json
✅ 2203.15556.rag.json: 223 chunks saved to 2203.15556.rag.chunks.json
✅ 2312.00752.rag.json: 436 chunks saved to 2312.00752.rag.chunks.json
✅ 2303.11312.rag.json: 229 chunks saved to 2303.11312.rag.chunks.json
✅ 2306.10209.rag.json: 182 chunks saved to 2306.10209.rag.chunks.json
✅ 2106.06967.rag.jso

5077